# IOAI — 2025 Team Selection Day4 Nlp Masked Word (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-team-selection-day4-nlp-masked-word'
for f in ['train.csv','val.csv','public_test.csv','sample_submission.csv']:
    if not os.path.exists(f): os.system(f'wget -q {BASE}/{f}')
print('데이터 준비:', [f for f in ['train.csv','val.csv','public_test.csv','sample_submission.csv'] if os.path.exists(f)])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Day 4 — Masked-Word Position · 모범답안 (Reference Solution)

> **Kazakhstan – Team Selection 2025 · Day 4**

Faithful reproduction of the strong approach (Batyr Yerdenov's TST solution): fine-tune **`bert-base-multilingual-cased`** as a **position classifier** (the masked sentence → softmax over indices 0…24), trained on synthetically-masked training sentences, with a **capitalisation post-rule** (lower-case start → index 0). Saves `submission_val.csv` (accuracy grading) and `submission.csv` (Kaggle).

In [ ]:
import os, pandas as pd, numpy as np, random
def _find(f):
    for b in [".","..","../..","/home/yhpark/ioai/problems/Kazakhstan-2025/Team-Selection/Day4_NLP_Masked_Word"]:
        if os.path.exists(os.path.join(b,f)): return b
    return "."
DATA=_find("train.csv")
train=pd.read_csv(os.path.join(DATA,"train.csv"))            # col: sentence (full)
val  =pd.read_csv(os.path.join(DATA,"val.csv"))             # ID, masked_sentence  (graded by accuracy)
test =pd.read_csv(os.path.join(DATA,"public_test.csv"))     # ID, masked_sentence  (Kaggle)
print("train",train.shape,"val",val.shape,"test",test.shape)


## Build masked training pairs

In [ ]:
random.seed(0)
MAXLEN=25
sent=train["sentence"].dropna().astype(str).tolist()
random.shuffle(sent); sent=sent[:8000]   # subsample for a fast, faithful demo
mtxt=[]; mlab=[]
for s in sent:
    w=s.split()
    if not (2<=len(w)<=MAXLEN): continue
    j=random.randrange(len(w))
    mtxt.append(" ".join(w[:j]+w[j+1:])); mlab.append(j)
print("training pairs:", len(mtxt))


## Fine-tune mBERT position classifier

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
device='cuda' if torch.cuda.is_available() else 'cpu'
MODEL="bert-base-multilingual-cased"
tok=AutoTokenizer.from_pretrained(MODEL)
model=AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=MAXLEN).to(device)

def encode(texts):
    return tok(list(texts), padding=True, truncation=True, max_length=48, return_tensors="pt")
enc=encode(mtxt); y=torch.tensor(mlab)
ds=TensorDataset(enc["input_ids"], enc["attention_mask"], y)
dl=DataLoader(ds, batch_size=32, shuffle=True)
opt=torch.optim.AdamW(model.parameters(), lr=3e-5)
model.train()
for ep in range(2):
    tot=0
    for ids,am,yb in dl:
        ids,am,yb=ids.to(device),am.to(device),yb.to(device)
        opt.zero_grad(); out=model(input_ids=ids, attention_mask=am, labels=yb)
        out.loss.backward(); opt.step(); tot+=out.loss.item()*len(yb)
    print(f"epoch {ep+1}  loss={tot/len(ds):.4f}")


## Predict (argmax + capitalisation rule) → submissions

In [ ]:
@torch.no_grad()
def predict(masked):
    model.eval(); out=[]
    for i in range(0,len(masked),64):
        chunk=[str(x) for x in masked[i:i+64]]
        e=encode(chunk); ids=e["input_ids"].to(device); am=e["attention_mask"].to(device)
        logit=model(input_ids=ids, attention_mask=am).logits.cpu().numpy()
        for s,lg in zip(chunk, logit):
            w=s.split()
            if w and w[0][:1].islower(): out.append(0); continue
            p=int(lg.argmax()); out.append(min(p, len(w)))   # clip to a valid insertion index
    return out
for df,outf in [(val,"submission_val.csv"),(test,"submission.csv")]:
    pd.DataFrame({"ID":df["ID"], "word_index":predict(list(df["masked_sentence"]))}).to_csv(outf,index=False)
print("wrote submission_val.csv / submission.csv")


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission_val.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)